## NbS river flood damages 

### Import relevant things and set base path

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt
import numpy as np
import rioxarray
from shapely.geometry import box
import matplotlib as mpl
from matplotlib.patches import Polygon, Rectangle, RegularPolygon
from rasterio.plot import plotting_extent
from matplotlib.colors import LogNorm, BoundaryNorm, ListedColormap, SymLogNorm                       # since you use LogNorm
from matplotlib.ticker import MaxNLocator, LogLocator, MultipleLocator, FixedLocator, FuncFormatter, NullLocator, ScalarFormatter
from rasterio.warp import calculate_default_transform, reproject, Resampling
from matplotlib import font_manager as fm
import matplotlib.patheffects as pe

from matplotlib.colors import Normalize


### Base and output paths

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
output_dir = base_path / "Outputs"
out_dir = base_path / "figures"

### Jamaica boundary 

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

In [ ]:
admin_boundary_path = base_path / "Inputs/Boundaries/admin_boundaries.gpkg"
admin1 = gpd.read_file(admin_boundary_path, layer="admin1")
admin1.crs
admin1.head()

## Figure helper 

In [ ]:
# TRUE scale bar in map units (needs projected CRS in meters)
def draw_scale_bar(ax, gdf, where="right-top", pad=0.04,
                   length_km="auto", max_frac=0.28,
                   lw=0.6, tick_h_frac=0.012,
                   fs_lab=6, fs_unit=6, unit_text="km"):
    crs = getattr(gdf, "crs", None)
    if crs is None or not crs.is_projected:
        return  # skip if not projected

    minx, miny, maxx, maxy = gdf.total_bounds
    W, H = (maxx - minx), (maxy - miny)

    # pleasant length that fits ≤ max_frac of map width
    if length_km == "auto":
        candidates = np.array([2, 5, 10, 20, 25, 50, 100], dtype=float)
        target = max_frac * (W/1000.0)
        valid = candidates[candidates <= max(1.0, target)]
        length_km = float(valid[-1]) if valid.size else 5.0
    L = length_km * 1000.0

    # anchor (x0,y0)
    x0 = minx + pad*W if "left"  in where else maxx - pad*W - L
    y0 = maxy - pad*H if "top"   in where else miny + pad*H

    # bar
    ax.plot([x0, x0+L], [y0, y0], color="black", lw=lw, clip_on=False)

    # ticks at 0, mid, end
    tick_h = tick_h_frac * H
    for xi in (x0, x0+L/2, x0+L):
        ax.plot([xi, xi], [y0 - tick_h/2, y0 + tick_h/2], color="black", lw=lw, clip_on=False)

    # numeric labels
    ylab = y0 - 2.1*tick_h
    ax.text(x0,     ylab, "0",                   ha="center", va="top", fontsize=fs_lab)
    ax.text(x0+L/2, ylab, f"{int(length_km//2)}",ha="center", va="top", fontsize=fs_lab)
    ax.text(x0+L,   ylab, f"{int(length_km)}",   ha="center", va="top", fontsize=fs_lab)
    ax.text(x0+L + 0.012*W, y0, unit_text, ha="left", va="center", fontsize=fs_unit)

    # return center-above point (DATA coords) for the arrow
    return (x0 + L/2, y0 + 2.2*tick_h)

def draw_north_arrow_axes(
    ax, x_ax, y_ax, *,
    size_frac=0.05,      # overall height in axes coords
    gap_frac=0.035,      # vertical gap above the bar
    shaft_w_frac=0.18,   # shaft width (fraction of size)
    head_w_frac=0.65,    # head width (fraction of size)
    head_h_frac=0.70,    # head/shaft split
    color="black", fs=6, lw=0.6
):
    """Neat 'N' + upright arrow in AXES coords (0–1)."""
    y0 = y_ax + gap_frac
    shaft_h = size_frac * (1 - head_h_frac)
    head_h  = size_frac * head_h_frac
    shaft_w = size_frac * shaft_w_frac
    head_w  = size_frac * head_w_frac

    # shaft
    rect = Rectangle((x_ax - shaft_w/2, y0), shaft_w, shaft_h,
                     transform=ax.transAxes, facecolor=color, edgecolor=color,
                     linewidth=lw, zorder=15, clip_on=False)
    ax.add_patch(rect)

    # triangle head
    tip_y = y0 + shaft_h + head_h
    tri = Polygon([(x_ax, tip_y),
                   (x_ax - head_w/2, y0 + shaft_h),
                   (x_ax + head_w/2, y0 + shaft_h)],
                  closed=True, transform=ax.transAxes,
                  facecolor=color, edgecolor=color,
                  linewidth=lw, zorder=15, clip_on=False)
    ax.add_patch(tri)

    # "N"
    ax.text(x_ax, tip_y + size_frac*0.28, "N",
            transform=ax.transAxes, ha="center", va="bottom",
            fontsize=fs, fontweight="bold", color=color)

In [ ]:
PREF = "Arial" if any(f.name == "Arial" for f in fm.fontManager.ttflist) else "Helvetica"

NATURE_RC = {
    # Output
    "figure.dpi": 300,         # on-screen
    "savefig.dpi": 600,        # PNG export (RGB)
    "savefig.facecolor": "white",
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",  # editable text

    # Typeface & sizes (5–7 pt at 90 mm)
    "font.family": PREF,
    "font.sans-serif": [PREF, "Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 6.5,
    "axes.titlesize": 7,
    "axes.labelsize": 6.5,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,

    # Greek letters to match sans (no LaTeX needed)
    "mathtext.fontset": "dejavusans",  # consistent Greek
    "axes.linewidth": 0.35,
}
# Apply globally, or keep using `with mpl.rc_context(NATURE_RC):` around each figure
mpl.rcParams.update(NATURE_RC)
print("Using font:", PREF)

def mm_to_in(mm): 
    return mm / 25.4

#### Read in basins 

In [ ]:
major_basins_plus_coastal = base_path / "major_basins_plus_coastal.gpkg"
major_basins_plus_coastal = gpd.read_file(major_basins_plus_coastal)

In [ ]:
# 0) One catchment per polygon, numeric ID 1..N
catchments = (
    major_basins_plus_coastal
      .loc[major_basins_plus_coastal.geometry.notna(), ["geometry"]]
      .reset_index(drop=True)
      .copy()
)
catchments["catchment_uid"] = catchments.index + 1

In [ ]:
catchments.head()

In [ ]:

# catchments_with_river = base_path / "catchments_with_river_for_eads.gpkg"
# catchments_with_river = gpd.read_file(catchments_with_river)
# catchments_with_river.head()

#### Read in damage files which underpin avoided damages by sector and subsector

In [ ]:
damage_future = base_path / "NbS_river_catchment/damage__future.parquet"
damage_future = pd.read_parquet(damage_future)

damage_future["avoided_ead"] = (
    damage_future["baseline__fluvial__ead"] - damage_future["future__fluvial__ead"]
)

damage_future.head()

#### Read in damage files which show where to restore forest 

In [ ]:
damage_reduction_min = base_path / "NbS_river_catchment/damage_reduction_min.tif"
damage_reduction_max = base_path / "NbS_river_catchment/damage_reduction_max.tif"

#### Read in avoided EAD files which show where the infrastructure damages are concentrated

In [ ]:
avoided_fluvial_ead_max_3448 = base_path / "NbS_river_catchment/avoided_fluvial_ead_max_3448.tif"

In [ ]:
avoided_fluvial_ead_max_300m_smoothed = base_path / "NbS_river_catchment/avoided__fluvial__ead_max_300m_smoothed.tif"

## Sector-level analysis 

#### Figure out what sector names we have 

In [ ]:
for val in sorted(damage_future["asset_class"].unique()):
    print(val)

In [ ]:
# mapping from asset_class to category
sector_map = {
    # Buildings
    "buildings_assigned_economic_activity_areas": "buildings",

    # Transport
    "rail_edges": "transport",
    "rail_nodes": "transport",
    "roads_edges": "transport",
    "roads_nodes": "transport",
    "airport_polygon_areas": "transport",
    "port_polygon_areas": "transport",  
    
    # Water
    "irrigation_assets_NIC_edges": "water",
    "irrigation_assets_NIC_nodes": "water",
    "pipelines_NWC_edges": "water",
    "potable_facilities_NWC_nodes": "water",
    "waste_water_facilities_NWC_nodes": "water",

    # Energy
    "electricity_network_v3.1_nodes": "energy",
}

In [ ]:
# add a new column with the category
damage_future["sector"] = damage_future["asset_class"].map(sector_map)

# now you can sum (or any other aggregation) by category
sector_sums = damage_future.groupby("sector")["avoided_ead"].sum().reset_index()

print(sector_sums)

In [ ]:
# Baseline EAD totals by sector
baseline_sector_sums = (
    damage_future
    .groupby("sector", dropna=False)["baseline__fluvial__ead"]
    .sum()
    .reset_index(name="baseline_ead")
)

print(baseline_sector_sums)

In [ ]:
# --- Sector rollups + TOTAL + pretty print (single cell) ---------------------
summary = (
    damage_future
    .assign(sector=lambda d: d["asset_class"].map(sector_map))
    .groupby("sector", dropna=False, as_index=False)
    .agg(
        baseline_ead=("baseline__fluvial__ead", "sum"),
        future_ead=("future__fluvial__ead", "sum"),
        avoided_ead=("avoided_ead", "sum"),
    )
    .assign(
        avoided_share=lambda d: np.where(
            d["baseline_ead"] > 0, d["avoided_ead"] / d["baseline_ead"], np.nan
        )
    )
)

# Optional simple tables (no TOTAL row)
sector_sums = summary[["sector", "avoided_ead"]].copy()
baseline_sector_sums = summary[["sector", "baseline_ead"]].copy()

# Shares of totals
baseline_total = summary["baseline_ead"].sum()
avoided_total  = summary["avoided_ead"].sum()

summary = summary.assign(
    share_of_total_baseline=np.where(
        baseline_total > 0, summary["baseline_ead"] / baseline_total, np.nan
    ),
    share_of_total_avoided=np.where(
        avoided_total > 0, summary["avoided_ead"] / avoided_total, np.nan
    ),
)

# TOTAL row
total_row = pd.DataFrame([{
    "sector": "TOTAL",
    "baseline_ead": baseline_total,
    "future_ead": summary["future_ead"].sum(),
    "avoided_ead": avoided_total,
    "avoided_share": (avoided_total / baseline_total) if baseline_total > 0 else np.nan,
    "share_of_total_baseline": 1.0 if baseline_total > 0 else np.nan,
    "share_of_total_avoided": 1.0 if avoided_total > 0 else np.nan,
}])

summary_with_total = pd.concat([summary, total_row], ignore_index=True)

# Pretty print (billions)
to_bil = ["baseline_ead", "future_ead", "avoided_ead"]
pretty = summary_with_total.assign(**{c: summary_with_total[c] / 1e9 for c in to_bil})

fmt = {
    "baseline_ead": "{:,.2f}".format,
    "future_ead": "{:,.2f}".format,
    "avoided_ead": "{:,.2f}".format,
    "avoided_share": "{:.1%}".format,
    "share_of_total_baseline": "{:.1%}".format,
    "share_of_total_avoided": "{:.1%}".format,
}

# Keep TOTAL last; sort sectors by baseline desc
is_total = pretty["sector"].eq("TOTAL")
to_print = pd.concat(
    [pretty.loc[~is_total].sort_values("baseline_ead", ascending=False),
     pretty.loc[is_total]],
    ignore_index=True
)

print(to_print.to_string(index=False, formatters=fmt))

In [ ]:

# --- display settings ---
CURRENCY = "J$"   # change to "USD" if you converted elsewhere
SCALE    = 1e9    # billions
UNIT_TAG = f"{CURRENCY} bn"

# Start from your existing 'summary_with_total'
pretty = summary_with_total.copy()

# Scale money columns to billions
money_cols = ["baseline_ead", "future_ead", "avoided_ead"]
for c in money_cols:
    pretty[c] = pretty[c] / SCALE

# Build a *print-only* copy with formatted strings + unit-tagged headers
to_print = pretty.copy()

# Format money + percent columns
to_print["baseline_ead"] = to_print["baseline_ead"].map(lambda v: f"{v:,.2f}")
to_print["future_ead"]   = to_print["future_ead"].map(lambda v: f"{v:,.2f}")
to_print["avoided_ead"]  = to_print["avoided_ead"].map(lambda v: f"{v:,.2f}")

# Percent columns may or may not be present depending on your earlier code
if "avoided_share" in to_print:
    to_print["avoided_share"] = to_print["avoided_share"].map(lambda v: f"{v:.1%}")

for pc in ("share_of_total_baseline", "share_of_total_avoided"):
    if pc in to_print:
        to_print[pc] = to_print[pc].apply(lambda x: "" if pd.isna(x) else f"{x:.1%}")

# Rename headers to include units
rename_map = {
    "baseline_ead": f"baseline_ead [{UNIT_TAG}]",
    "future_ead":   f"future_ead [{UNIT_TAG}]",
    "avoided_ead":  f"avoided_ead [{UNIT_TAG}]",
}
to_print = to_print.rename(columns=rename_map)

print(to_print.to_string(index=False))
print(f"\nUnits: {UNIT_TAG} (1 {UNIT_TAG.split()[1]} = 1e9).")

In [ ]:
# --- Convert J$ → USD and pretty print in USD millions ---


JMD_PER_USD = 150.0
USD_PER_JMD = 1.0 / JMD_PER_USD
MILLIONS    = 1e6

money_cols = ["baseline_ead", "future_ead", "avoided_ead"]

# 1) Convert currency
summary_usd = summary_with_total.copy()
for c in money_cols:
    summary_usd[c] = summary_usd[c] * USD_PER_JMD  # J$ → USD

# 2) Scale to millions for display
pretty_usd_m = summary_usd.copy()
for c in money_cols:
    pretty_usd_m[c] = pretty_usd_m[c] / MILLIONS   # USD → USD millions

# 3) Build a print-only view with formatting
to_print_usd_m = pretty_usd_m.copy()

fmt_num = lambda v: "" if pd.isna(v) else f"{v:,.2f}"
for c in money_cols:
    to_print_usd_m[c] = to_print_usd_m[c].map(fmt_num)

if "avoided_share" in to_print_usd_m:
    to_print_usd_m["avoided_share"] = to_print_usd_m["avoided_share"].apply(
        lambda x: "" if pd.isna(x) else f"{x:.1%}"
    )

for pc in ("share_of_total_baseline", "share_of_total_avoided"):
    if pc in to_print_usd_m:
        to_print_usd_m[pc] = to_print_usd_m[pc].apply(
            lambda x: "" if pd.isna(x) else f"{x:.1%}"
        )

# 4) Rename headers to include units
to_print_usd_m = to_print_usd_m.rename(columns={
    "baseline_ead": "baseline_ead [USD mn]",
    "future_ead":   "future_ead [USD mn]",
    "avoided_ead":  "avoided_ead [USD mn]",
})

print(to_print_usd_m.to_string(index=False))
print("\nAssumed FX: 1 USD = 150 J$; values shown in USD millions.")

In [ ]:
# FX and scaling
JMD_PER_USD = 150.0
USD_PER_JMD = 1.0 / JMD_PER_USD
SCALE_MN    = 1e6

money_cols = ["baseline_ead", "future_ead", "avoided_ead"]

# 0) Convert J$ → USD, then scale to millions
plot_usd = summary_with_total.copy()
for c in money_cols:
    plot_usd[c] = plot_usd[c] * USD_PER_JMD / SCALE_MN   # USD millions

# 1) Build plotting table (baseline & avoided only)
df = plot_usd[["sector", "baseline_ead", "avoided_ead"]].copy()
df = df[df["sector"].notna()]  # drop NaNs if any

# Put TOTAL at the end, sort others by baseline desc
order = (
    df.query('sector != "TOTAL"')
      .sort_values("baseline_ead", ascending=False)["sector"]
      .tolist()
    + ["TOTAL"]
)
df = df.set_index("sector").loc[order].reset_index()

# Values for stacked plotting
df["_baseline"] = df["baseline_ead"]
df["_avoided"]  = np.clip(df["avoided_ead"], 0, df["_baseline"])   # cap avoided ≤ baseline
df["_bottom"]   = df["_baseline"] - df["_avoided"]

x = np.arange(len(df))
is_total = df["sector"].eq("TOTAL").to_numpy()

# 2) Plot
fig, ax = plt.subplots(figsize=(8.5, 4.8))

# baseline bars (darker for TOTAL)
base_colors = np.where(is_total, "#B0B0B0", "#C8C8C8")
ax.bar(x, df["_baseline"], color=base_colors, edgecolor="#333333", linewidth=0.6, label="Baseline EAD")

# avoided portion as a hatched top segment
ax.bar(x, df["_avoided"], bottom=df["_bottom"],
       facecolor="none", edgecolor="#2E7D32", linewidth=0.9,
       hatch="///", label="Avoided through forest restoration", zorder=3)

# thin separator before TOTAL
if len(df) > 1:
    ax.axvline(len(df) - 1.5, linestyle=":", color="0.5", linewidth=0.8)

# y-grid (optional, subtle)
ax.grid(axis="y", linestyle=":", color="0.88", zorder=0)
ax.set_axisbelow(True)

# x/y labels & title
ax.set_xticks(x)
ax.set_xticklabels(df["sector"], rotation=0)
ax.set_ylabel("EADs (US$ million)")
ax.yaxis.set_major_locator(MultipleLocator(100))                 # or 50, etc.
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:,.0f}"))
ax.set_title("Expected Annual Damages (EADs) and % avoided through forest restoration",
             fontweight="bold")
# percent labels
ylim = ax.get_ylim()
offset = 0.01 * (ylim[1] - ylim[0])                 # small vertical offset in data units
threshold = 0.05 * df["_baseline"].max()            # 5% of max bar height


# sectors whose % label should be outside (above the bar)
force_outside = {"buildings", "transport", "TOTAL"}
ax.margins(y=0.06)  # a bit of top margin for outside labels

for i, r in df.iterrows():
    if r["_baseline"] <= 0 or r["_avoided"] <= 0:
        continue
    pct = r["_avoided"] / r["_baseline"]
    top = r["_bottom"] + r["_avoided"]

    want_outside = (r["sector"] in force_outside) or (r["_avoided"] < threshold)
    if want_outside:
        y, va = top + offset, "bottom"     # outside (above the bar)
    else:
        y, va = r["_bottom"] + r["_avoided"] / 2, "center"  # inside
    ax.text(x[i], y, f"{pct:.0%}", ha="center", va=va, fontsize=9,
            color="#2E7D32", clip_on=False, zorder=5)
    
# legend outside to keep the plot area clean
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
fig.subplots_adjust(right=0.83)  # room for the outside legend

fig.tight_layout()

# ---- save BEFORE show ----
# Reuse your existing 'out_dir' if defined; otherwise create a sensible default.
try:
    out_dir
except NameError:
    base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
    out_dir = base_path / "figures"

out_dir.mkdir(parents=True, exist_ok=True)
fname = out_dir / "national_avoided_EADs_usd_mn"
fig.savefig(fname.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(fname.with_suffix(".pdf"),              bbox_inches="tight")

plt.show()
print("Saved to:", fname.with_suffix(".png"), "and", fname.with_suffix(".pdf"))
print("Assumed FX: 1 USD = 150 J$; values are in USD millions.")

In [ ]:
# Helper to make JMD human-readable
def humanize_jmd(x):
    x = float(x)
    if abs(x) >= 1e9:
        return f"J${x/1e9:.2f} bn"
    elif abs(x) >= 1e6:
        return f"J${x/1e6:.2f} m"
    else:
        return f"J${x:,.0f}"

# 1) Group & sum
sector_sums = (damage_future
               .groupby("sector", dropna=False, as_index=False)
               .agg(avoided_ead_JMD=("avoided_ead", "sum")))

# Optional: label any unmapped rows instead of NaN
sector_sums["sector"] = sector_sums["sector"].fillna("unknown")

# 2) Add millions, share of total, readable label
total_jmd = sector_sums["avoided_ead_JMD"].sum()
sector_sums["avoided_ead_million_JMD"] = sector_sums["avoided_ead_JMD"] / 1e6
sector_sums["readable"] = sector_sums["avoided_ead_JMD"].apply(humanize_jmd)
sector_sums["share_pct"] = 100 * sector_sums["avoided_ead_JMD"] / total_jmd

# 3) Pretty display (rounded) and TOTAL row
display_df = (sector_sums
              .sort_values("avoided_ead_JMD", ascending=False)
              .assign(
                  avoided_ead_million_JMD=lambda d: d["avoided_ead_million_JMD"].round(1),
                  share_pct=lambda d: d["share_pct"].round(1),
              ))

total_row = pd.DataFrame([{
    "sector": "TOTAL",
    "avoided_ead_JMD": total_jmd,
    "avoided_ead_million_JMD": round(total_jmd / 1e6, 1),
    "readable": humanize_jmd(total_jmd),
    "share_pct": 100.0,
}])

sector_with_total = pd.concat([display_df, total_row], ignore_index=True)

print(sector_with_total.to_string(index=False))

# DAMAGES BY RETURN PERIOD (RP)

In [ ]:
# TRY LOOK BY RP

import re

# Ensure sector exists
damage_future["sector"] = damage_future["asset_class"].map(sector_map)

# Keep just the sectors you care about (optional)
wanted_sectors = ["buildings", "energy", "transport", "water"]
df = damage_future[damage_future["sector"].isin(wanted_sectors)].copy()

# --- 1) detect RP pairs present ---
rp_levels = sorted(
    int(m.group(1))
    for col in df.columns
    if (m := re.match(r"baseline__fluvial__rp_(\d+)$", col))
    and f"future__fluvial__rp_{m.group(1)}" in df.columns
)

if not rp_levels:
    raise ValueError("No matching baseline/future RP columns found.")

# --- 2) build tidy (long) table across RPs ---
records = []
for rp in rp_levels:
    bcol = f"baseline__fluvial__rp_{rp}"
    fcol = f"future__fluvial__rp_{rp}"
    tmp = df[["sector", bcol, fcol]].copy()
    tmp.rename(columns={bcol: "baseline", fcol: "future"}, inplace=True)
    tmp["rp"] = rp
    tmp["avoided"] = tmp["baseline"] - tmp["future"]
    records.append(tmp)

rp_long = pd.concat(records, ignore_index=True)

# --- 3) aggregate by sector & RP ---
sector_rp = (
    rp_long
    .groupby(["sector", "rp"], dropna=False)
    .agg(
        baseline=("baseline", "sum"),
        future=("future", "sum"),
        avoided=("avoided", "sum"),
    )
    .reset_index()
)

# avoided_share within each sector & RP
sector_rp["avoided_share"] = np.where(
    sector_rp["baseline"] > 0,
    sector_rp["avoided"] / sector_rp["baseline"],
    np.nan
)

# --- 4) add TOTAL row per RP ---
totals = (
    rp_long
    .groupby("rp", as_index=False)
    .agg(
        baseline=("baseline", "sum"),
        future=("future", "sum"),
        avoided=("avoided", "sum"),
    )
)
totals["sector"] = "TOTAL"
totals["avoided_share"] = np.where(
    totals["baseline"] > 0,
    totals["avoided"] / totals["baseline"],
    np.nan
)

summary_rp = pd.concat([sector_rp, totals], ignore_index=True)

# --- 5) add each sector's share of the RP totals (baseline/avoided) ---
# (excluding the TOTAL row when computing shares)
is_total = summary_rp["sector"].eq("TOTAL")
summary_rp["baseline_total_rp"] = summary_rp.groupby("rp")["baseline"].transform("sum")
summary_rp["avoided_total_rp"]  = summary_rp.groupby("rp")["avoided"].transform("sum")

summary_rp.loc[~is_total, "share_of_total_baseline"] = np.where(
    summary_rp.loc[~is_total, "baseline_total_rp"] > 0,
    summary_rp.loc[~is_total, "baseline"] / summary_rp.loc[~is_total, "baseline_total_rp"],
    np.nan
)
summary_rp.loc[~is_total, "share_of_total_avoided"] = np.where(
    summary_rp.loc[~is_total, "avoided_total_rp"] > 0,
    summary_rp.loc[~is_total, "avoided"] / summary_rp.loc[~is_total, "avoided_total_rp"],
    np.nan
)

# --- 6) present in billions with explicit units ---
SCALE = 1e9
UNIT  = "J$ bn"

money_cols = ["baseline", "future", "avoided"]
summary_rp_bil = summary_rp.copy()
for c in money_cols:
    summary_rp_bil[c] = summary_rp_bil[c] / SCALE

# Nice printout
def _fmt_df(d):
    d = d.copy()
    for c in money_cols:
        d[c] = d[c].map(lambda v: f"{v:,.2f}")
    d["avoided_share"] = d["avoided_share"].map(lambda v: f"{v:.1%}" if pd.notna(v) else "")
    for c in ["share_of_total_baseline", "share_of_total_avoided"]:
        if c in d:
            d[c] = d[c].map(lambda v: f"{v:.1%}" if pd.notna(v) else "")
    d = d.rename(columns={
        "baseline": f"baseline [{UNIT}]",
        "future":   f"future [{UNIT}]",
        "avoided":  f"avoided [{UNIT}]",
    })
    return d[["sector", "rp", f"baseline [{UNIT}]", f"future [{UNIT}]", f"avoided [{UNIT}]",
              "avoided_share", "share_of_total_baseline", "share_of_total_avoided"]]

print(_fmt_df(summary_rp_bil).sort_values(["sector","rp"]).to_string(index=False))
print("\nUnits:", UNIT)

# --- 7) (Optional) quick pivots for comparison across RPs ---
avoided_pivot = (
    summary_rp_bil
    .pivot_table(index="sector", columns="rp", values="avoided", aggfunc="sum")
    .reindex(wanted_sectors + ["TOTAL"])
)
avoided_share_pivot = (
    summary_rp
    .pivot_table(index="sector", columns="rp", values="avoided_share", aggfunc="mean")
    .reindex(wanted_sectors + ["TOTAL"])
)

# Format for display (optional)
print("\nAvoided damage by RP [J$ bn]:")
print(avoided_pivot.applymap(lambda v: f"{v:,.2f}").to_string())

print("\nAvoided share by RP [% of baseline at that RP]:")
print(avoided_share_pivot.applymap(lambda v: "" if pd.isna(v) else f"{v:.1%}").to_string())

## Sub-sector analysis

In [ ]:
transport_subsector_map = {
    "rail_edges": "rail",
    "rail_nodes": "rail",
    "roads_edges": "roads",
    "roads_nodes": "roads",
    "airport_polygon_areas": "airports",
    "port_polygon_areas": "ports",  
}

In [ ]:
# Add subsector labels
damage_future["transport_subsector"] = damage_future["asset_class"].map(transport_subsector_map)

# Keep only rows that mapped to a transport subsector
df_t = damage_future.loc[damage_future["transport_subsector"].notna()].copy()

# Group & sum
transport_sums = (df_t.groupby("transport_subsector", as_index=False)
                    .agg(avoided_ead_JMD=("avoided_ead", "sum")))

# Add millions, shares, readable labels
total_t = transport_sums["avoided_ead_JMD"].sum()
transport_sums["avoided_ead_million_JMD"] = transport_sums["avoided_ead_JMD"] / 1e6
transport_sums["share_pct"] = 100 * transport_sums["avoided_ead_JMD"] / total_t
transport_sums["readable"] = transport_sums["avoided_ead_JMD"].apply(humanize_jmd)

# Pretty display + TOTAL row
display_df = (transport_sums
              .sort_values("avoided_ead_JMD", ascending=False)
              .assign(
                  avoided_ead_million_JMD=lambda d: d["avoided_ead_million_JMD"].round(1),
                  share_pct=lambda d: d["share_pct"].round(1),
              ))

total_row = pd.DataFrame([{
    "transport_subsector": "TOTAL (transport)",
    "avoided_ead_JMD": total_t,
    "avoided_ead_million_JMD": round(total_t / 1e6, 1),
    "share_pct": 100.0,
    "readable": humanize_jmd(total_t),
}])

transport_with_total = pd.concat([display_df, total_row], ignore_index=True)

print(transport_with_total.to_string(index=False))

In [ ]:
water_subsector_map = {
    "irrigation_assets_NIC_edges": "irrigation",
    "irrigation_assets_NIC_nodes": "irrigation",
    "pipelines_NWC_edges": "pipelines",
    "potable_facilities_NWC_nodes": "potable",
    "waste_water_facilities_NWC_nodes": "wastewater",
}

# label subsectors
damage_future["water_subsector"] = damage_future["asset_class"].map(water_subsector_map)

# keep only mapped rows
df_w = damage_future.loc[damage_future["water_subsector"].notna()].copy()

# group & sum
water_sums = (df_w.groupby("water_subsector", as_index=False)
                .agg(avoided_ead_JMD=("avoided_ead", "sum")))

# add millions, shares, readable
total_w = water_sums["avoided_ead_JMD"].sum()
water_sums["avoided_ead_million_JMD"] = water_sums["avoided_ead_JMD"] / 1e6
water_sums["share_pct"] = 100 * water_sums["avoided_ead_JMD"] / total_w
water_sums["readable"] = water_sums["avoided_ead_JMD"].apply(humanize_jmd)

# pretty display + TOTAL row
display_df = (water_sums.sort_values("avoided_ead_JMD", ascending=False)
                        .assign(
                            avoided_ead_million_JMD=lambda d: d["avoided_ead_million_JMD"].round(1),
                            share_pct=lambda d: d["share_pct"].round(1),
                        ))

total_row = pd.DataFrame([{
    "water_subsector": "TOTAL (water)",
    "avoided_ead_JMD": total_w,
    "avoided_ead_million_JMD": round(total_w / 1e6, 1),
    "share_pct": 100.0,
    "readable": humanize_jmd(total_w),
}])

water_with_total = pd.concat([display_df, total_row], ignore_index=True)

print(water_with_total.to_string(index=False))

# optional: list any water rows that didn't map to a subsector
unmapped_water = damage_future.loc[
    (damage_future["sector"] == "water") & (damage_future["water_subsector"].isna()),
    "asset_class"
].unique()
if len(unmapped_water):
    print("Unmapped water asset_class values:", unmapped_water)

In [ ]:
# sec = sector_with_total[sector_with_total["sector"] != "TOTAL"].copy()
# tr  = transport_with_total[transport_with_total["transport_subsector"] != "TOTAL (transport)"].copy()
# wa  = water_with_total[water_with_total["water_subsector"]     != "TOTAL (water)"].copy()

# sec["val"] = sec["avoided_ead_JMD"] / 1e9    # billions JMD
# tr["val"]  = tr["avoided_ead_JMD"] / 1e6     # millions JMD
# wa["val"]  = wa["avoided_ead_JMD"] / 1e6     # millions JMD

# # sort so largest ends up at the top
# sec = sec.sort_values("val", ascending=True)
# tr  = tr.sort_values("val",  ascending=True)
# wa  = wa.sort_values("val",  ascending=True)

# # ---------- figure (double-column width 180 mm, height <= 170 mm) ----------
# fig = plt.figure(figsize=(mm_to_in(180), mm_to_in(165)))
# gs = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[1.2, 1, 1], hspace=0.55)

# ax1 = fig.add_subplot(gs[0])
# ax2 = fig.add_subplot(gs[1])
# ax3 = fig.add_subplot(gs[2])

# def style_axis(ax):
#     ax.spines["top"].set_visible(False)
#     ax.spines["right"].set_visible(False)
#     ax.grid(axis="x", linestyle=":", linewidth=0.5, alpha=0.5)
#     ax.tick_params(axis="both", which="both", length=2)

# # neutral grayscale palette
# c1, c2, c3 = "#4a4a4a", "#6f6f6f", "#8c8c8c"

# # ---------- (a) Sectors ----------
# ax1.barh(sec["sector"], sec["val"], color=c1, edgecolor="black", linewidth=0.4)
# for y, v in enumerate(sec["val"].to_numpy()):
#     ax1.text(v, y, f" {v:.2f} bn", va="center", ha="left")
# ax1.set_xlabel("Avoided EAD (JMD billions)")
# ax1.set_title("Avoided EAD by sector", pad=6)
# style_axis(ax1)
# ax1.text(-0.02, 1.02, "a", transform=ax1.transAxes, fontweight="bold", va="bottom")

# # ---------- (b) Transport subsectors ----------
# ax2.barh(tr["transport_subsector"], tr["val"], color=c2, edgecolor="black", linewidth=0.4)
# for y, v in enumerate(tr["val"].to_numpy()):
#     ax2.text(v, y, f" {v:,.0f} M", va="center", ha="left")
# ax2.set_xlabel("Avoided EAD (JMD millions)")
# ax2.set_title("Transport subsectors", pad=6)
# style_axis(ax2)
# ax2.text(-0.02, 1.02, "b", transform=ax2.transAxes, fontweight="bold", va="bottom")

# # ---------- (c) Water subsectors ----------
# ax3.barh(wa["water_subsector"], wa["val"], color=c3, edgecolor="black", linewidth=0.4)
# for y, v in enumerate(wa["val"].to_numpy()):
#     ax3.text(v, y, f" {v:,.0f} M", va="center", ha="left")
# ax3.set_xlabel("Avoided EAD (JMD millions)")
# ax3.set_title("Water subsectors", pad=6)
# style_axis(ax3)
# ax3.text(-0.02, 1.02, "c", transform=ax3.transAxes, fontweight="bold", va="bottom")

# fig.tight_layout()

# # High-res export (RGB / vector)
# fig.savefig(base_path / "fig_avoided_EAD_panels.png", bbox_inches="tight")  # 600 dpi from rcParams if set
# fig.savefig(base_path / "fig_avoided_EAD_panels.pdf", bbox_inches="tight")
# plt.show()

### Visualising damages

In [ ]:
with rasterio.open(damage_reduction_min) as src:
    print("CRS:", src.crs)
    print("dtype:", src.dtypes[0])
    print("NoData:", src.nodata)
    arr = src.read(1, masked=True)  # respect NoData

print("shape:", arr.shape)
print("min/max:", arr.min(), arr.max())
print("non-zero count:", np.count_nonzero(arr))
print("unique (up to 10):", np.unique(arr.compressed())[:10])

In [ ]:
with rasterio.open(damage_reduction_max) as src:
    print("CRS:", src.crs)
    print("dtype:", src.dtypes[0])
    print("NoData:", src.nodata)
    arr_max = src.read(1, masked=True)  # respect NoData

In [ ]:
# --- Fig 3(c) — log scale, using rcParams for font sizes --------------------
import numpy as np, geopandas as gpd, matplotlib as mpl, matplotlib.pyplot as plt, rasterio
from rasterio.plot import plotting_extent
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter, NullLocator

with rasterio.open(damage_reduction_max) as src:
    arr    = src.read(1, masked=True)      # respects NoData
    extent = plotting_extent(src)

# data → billions; mask nonpositives
data_bil = np.array(arr, dtype="float64") / 1e9
data_bil[arr.mask | (data_bil <= 0)] = np.nan

# robust bounds (2–98%) for LogNorm
tiny = 1e-9
pos = np.asarray(data_bil[(~np.isnan(data_bil)) & (data_bil > 0)])
if pos.size == 0:
    raise ValueError("No positive values to plot (all zeros/NaNs).")

lo, hi = np.percentile(pos, [2, 98])
lo = max(float(lo), tiny)
hi = float(hi if hi > lo else lo * 1.01)

norm = LogNorm(vmin=lo, vmax=hi)  # single source of truth

# colormap
cmap = mpl.colormaps["Greens"].copy()
cmap.set_bad((0, 0, 0, 0))  # transparent NoData

# choose human-friendly units given vmax (still in billions)
def pick_unit_for_billions(hi_bil: float):
    if hi_bil >= 0.5:      return 1.0,  "J$ billions"   # keep billions
    if hi_bil >= 0.005:    return 1e3, "J$ millions"    # B → M
    return 1e6, "J$ thousands"                          # B → k

scale, unit_label = pick_unit_for_billions(hi)

with mpl.rc_context(NATURE_RC):
    # pull sizes from rcParams so you don't need hardcoded TITLE_FS/LABEL_FS/TICK_FS
    TITLE_FS = mpl.rcParams.get("figure.titlesize", 7)
    LABEL_FS = mpl.rcParams.get("axes.labelsize", 6)
    TICK_FS  = mpl.rcParams.get("xtick.labelsize", 5.5)

    fig, ax = plt.subplots(figsize=(mm_to_in(90), mm_to_in(60)))
    ax.set_axis_off()

    # raster
    im = ax.imshow(data_bil, norm=norm, cmap=cmap, extent=extent, origin="upper", interpolation="nearest")

    # Jamaica outline (white casing + black)
    try:
        outline_geom = jamaica_boundary.union_all()
    except AttributeError:
        outline_geom = jamaica_boundary.unary_union
    outline_gdf = gpd.GeoSeries([outline_geom], crs=jamaica_boundary.crs)
    outline_gdf.plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.0, zorder=5)
    outline_gdf.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.45, zorder=6)

    # --- Colorbar: decade ticks, readable labels in chosen units ---
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    # powers of 10 within [lo, hi]
    lo_pow = int(np.floor(np.log10(lo)))
    hi_pow = int(np.ceil(np.log10(hi)))
    ticks = (10.0 ** np.arange(lo_pow, hi_pow + 1))
    ticks = ticks[(ticks >= lo) & (ticks <= hi)]
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.015, ticks=ticks)

    def tick_fmt(x, _):
        v = x * scale  # convert from billions to chosen unit
        if scale == 1.0:         # billions
            return f"{v:.2f}" if v < 10 else f"{v:.1f}"
        if scale == 1e3:         # millions
            return f"{v:.1f}" if v < 10 else f"{v:.0f}"
        else:                    # thousands
            return f"{v:.2f}" if v < 1 else (f"{v:.1f}" if v < 10 else f"{v:.0f}")
    cbar.ax.yaxis.set_major_formatter(FuncFormatter(tick_fmt))
    cbar.minorticks_off()
    cbar.ax.yaxis.set_minor_locator(NullLocator())
    cbar.ax.tick_params(labelsize=TICK_FS, width=0.35, length=2)
    cbar.set_label(f"Avoided damages ({unit_label})", fontsize=LABEL_FS)

    # --- Scale bar + north arrow (top-right) ---
    pt = draw_scale_bar(
        ax, outline_gdf, where="right-top",
        pad=0.07, length_km="auto", max_frac=0.22,
        lw=0.5, tick_h_frac=0.012, fs_lab=5, fs_unit=5, unit_text="km"
    )
    if pt is not None:
        cx_data, cy_data = pt
        cx_ax, cy_ax = ax.transAxes.inverted().transform(ax.transData.transform((cx_data, cy_data)))
        draw_north_arrow_axes(
            ax, cx_ax, cy_ax,
            size_frac=0.080, gap_frac=0.050,
            shaft_w_frac=0.10, head_w_frac=0.30, head_h_frac=0.50,
            fs=5, lw=0.5
        )

    # higher title (use suptitle)
    fig.subplots_adjust(top=0.96)  # leave a bit more headroom
    fig.suptitle(
        "Fig 3(c) Candidates for forest restoration with highest avoided\nexpected annual damages (log scale)",
        fontsize=TITLE_FS, y=0.985
    )

    plt.show()
    out_png = output_dir / "forest_avoided_damages_log_thousands.png"
    out_pdf = output_dir / "forest_avoided_damages_log_thousands.pdf"
    fig.savefig(out_png, dpi=600, bbox_inches="tight")  # RGB, ≥300 dpi
    fig.savefig(out_pdf, bbox_inches="tight")           # vector
    print("Saved:", out_png, "and", out_pdf)

In [ ]:
# === Fig 3(c): Candidates with highest avoided damages — LINEAR SCALE =======
# choose human-friendly units given vmax (data are currently in BILLIONS)
def pick_unit_for_billions(hi_bil: float):
    if hi_bil >= 0.5:   return 1.0,  "J$ billions"   # keep billions
    if hi_bil >= 0.005: return 1e3, "J$ millions"    # B → M
    return 1e6, "J$ thousands"                       # B → k

# --- read raster
with rasterio.open(damage_reduction_max) as src:
    arr    = src.read(1, masked=True)       # respects NoData
    extent = plotting_extent(src)

# to billions; mask nonpositives
data_bil = np.array(arr, dtype="float64") / 1e9
data_bil[arr.mask | (data_bil <= 0)] = np.nan

pos = data_bil[~np.isnan(data_bil)]
if pos.size == 0:
    raise ValueError("No positive values to plot (all zeros/NaNs).")

# robust upper bound for linear scale (2–98th pct, then map 0..hi)
_, hi = np.percentile(pos, [2, 98])
hi = float(hi if hi > 0 else np.nanmax(pos))
if not np.isfinite(hi): hi = 1.0

# normalization (LINEAR)
norm = Normalize(vmin=0.0, vmax=hi)

# colormap (keep NoData transparent)
cmap = mpl.colormaps["Greens"].copy()
cmap.set_bad((0, 0, 0, 0))

# units for colorbar
scale, unit_label = pick_unit_for_billions(hi)

with mpl.rc_context(NATURE_RC):
    # pull sizes from your rcParams so this figure uses your global sizing
    TITLE_FS = mpl.rcParams.get("figure.titlesize", 7)
    LABEL_FS = mpl.rcParams.get("axes.labelsize", 6)
    # prefer ytick size; fall back to xtick or a small default
    TICK_FS  = mpl.rcParams.get("ytick.labelsize",
                    mpl.rcParams.get("xtick.labelsize", 5.5))

    fig, ax = plt.subplots(figsize=(mm_to_in(90), mm_to_in(60)))
    ax.set_axis_off()

    # raster
    im = ax.imshow(data_bil, norm=norm, cmap=cmap, extent=extent,
                   origin="upper", interpolation="nearest")

    # Jamaica outline (white casing + black)
    try:
        outline_geom = jamaica_boundary.union_all()
    except AttributeError:
        outline_geom = jamaica_boundary.unary_union
    outline_gdf = gpd.GeoSeries([outline_geom], crs=jamaica_boundary.crs)
    outline_gdf.plot(ax=ax, facecolor="none", edgecolor="white", linewidth=1.0, zorder=5)
    outline_gdf.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.45, zorder=6)

    # --- Colorbar: nice rounded ticks in chosen units
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.015)
    cbar.ax.yaxis.set_minor_locator(NullLocator())
    cbar.outline.set_linewidth(0.35)

    hi_scaled = hi * scale
    locator = MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10], min_n_ticks=4)
    ticks_scaled = locator.tick_values(0, hi_scaled)
    ticks_scaled = ticks_scaled[(ticks_scaled >= 0) & (ticks_scaled <= hi_scaled + 1e-12)]
    ticks_raw = ticks_scaled / scale
    cbar.set_ticks(ticks_raw)

    def tick_fmt(val_raw):
        y = val_raw * scale
        if y < 1:     s = f"{y:.2f}"
        elif y < 10:  s = f"{y:.1f}"
        else:         s = f"{y:.0f}"
        return s.rstrip("0").rstrip(".")
    cbar.set_ticklabels([tick_fmt(t) for t in ticks_raw])
    cbar.set_label(f"Avoided damages ({unit_label})", fontsize=LABEL_FS)
    cbar.ax.tick_params(labelsize=TICK_FS, width=0.35, length=2)

    # --- Scale bar + north arrow (top-right)
    pt = draw_scale_bar(
        ax, outline_gdf, where="right-top",
        pad=0.07, length_km="auto", max_frac=0.22,
        lw=0.5, tick_h_frac=0.012, fs_lab=5, fs_unit=5, unit_text="km"
    )
    if pt is not None:
        cx_data, cy_data = pt
        cx_ax, cy_ax = ax.transAxes.inverted().transform(ax.transData.transform((cx_data, cy_data)))
        draw_north_arrow_axes(
            ax, cx_ax, cy_ax,
            size_frac=0.080, gap_frac=0.050,
            shaft_w_frac=0.10, head_w_frac=0.30, head_h_frac=0.50,
            fs=5, lw=0.5
        )

    # title + save
    fig.subplots_adjust(top=0.965)
    fig.suptitle("Fig 3(b) Candidates for forest restoration based on avoided EADs",
                 fontsize=TITLE_FS, y=0.985)

    out_png = output_dir / "forest_avoided_damages_linear.png"
    out_pdf = output_dir / "forest_avoided_damages_linear.pdf"
    fig.savefig(out_png, dpi=600, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png, "and", out_pdf)

# If the mid-tones look a bit flat on linear, you can swap the `Normalize` line for:
# from matplotlib.colors import PowerNorm
# norm = PowerNorm(gamma=0.6, vmin=0.0, vmax=hi)  # NOT log; just brightens mid-range

### By catchments

In [ ]:
# 1) Zonal sum helper (sums raster values within each polygon; no area calc)
def zonal_sum_only(raster_path, gdf, id_col="catchment_uid", all_touched=False):
    rows = []
    with rasterio.open(raster_path) as src:
        gdf_proj = gdf.to_crs(src.crs).copy()
        gdf_proj["geometry"] = gdf_proj.geometry.buffer(0)  # repair invalid geoms
        rb = box(*src.bounds)
        gdf_proj = gdf_proj[gdf_proj.intersects(rb)].copy()

        nd = src.nodata
        scale = (src.scales[0] if getattr(src, "scales", None) else 1.0) or 1.0
        offset = (src.offsets[0] if getattr(src, "offsets", None) else 0.0) or 0.0

        for _, r in gdf_proj.iterrows():
            try:
                data, _ = mask(src, [r.geometry.__geo_interface__],
                               crop=True, filled=False, all_touched=all_touched)
            except ValueError:
                rows.append({id_col: r[id_col], "sum": 0.0})
                continue

            band = data[0].astype("float64")
            band = band * scale + offset  # apply scale/offset if present

            ma = np.ma.array(band, mask=np.ma.getmaskarray(band))  # keep outside masked
            if nd is not None:
                ma = np.ma.masked_where(band == nd, ma)
            ma = np.ma.masked_invalid(ma)

            rows.append({id_col: r[id_col], "sum": float(ma.sum()) if ma.count() else 0.0})

    out = pd.DataFrame(rows)
    # ensure every ID appears (zeros for non-overlapping polygons)
    all_ids = gdf[[id_col]].copy()
    out = all_ids.merge(out, on=id_col, how="left").fillna({"sum": 0.0})
    return out


In [ ]:
# 2) Compute avoided_ead sums (min/max) by numeric catchment_uid
zs_min = zonal_sum_only(damage_reduction_min, catchments, id_col="catchment_uid") \
           .rename(columns={"sum": "avoided_ead_min"})
zs_max = zonal_sum_only(damage_reduction_max, catchments, id_col="catchment_uid") \
           .rename(columns={"sum": "avoided_ead_max"})

result = (catchments.merge(zs_min, on="catchment_uid")
                   .merge(zs_max, on="catchment_uid"))

# Optional central estimate
result["avoided_ead_mid"] = result[["avoided_ead_min","avoided_ead_max"]].mean(axis=1, skipna=True)






In [ ]:
# --- AREA PER CATCHMENT (km²) -----------------------------------------------
result = result.copy()
result["area_km2"] = result.geometry.area / 1e6  # assumes current CRS is metres

# Rebuild your table (sorted by avoided_ead_max) and show top 30 with area
tbl = (result.drop(columns="geometry")
             .sort_values("avoided_ead_max", ascending=False))

# Put key columns first (keep any others that exist)
first = ["catchment_uid", "area_km2", "avoided_ead_min", "avoided_ead_max", "avoided_ead_mid",
         "avoided_ead_min_usd_mn", "avoided_ead_max_usd_mn", "avoided_ead_mid_usd_mn"]
cols = [c for c in first if c in tbl.columns] + [c for c in tbl.columns if c not in first]
tbl = tbl[cols]

pd.options.display.float_format = "{:,.2f}".format
print("Columns:", print(tbl.columns))
print(tbl.head(30).to_string(index=False))

In [ ]:
# 3) Table view, sorted by avoided_ead_max
tbl = (result.drop(columns="geometry")
             .sort_values("avoided_ead_max", ascending=False))
print("Columns:", list(tbl.columns))
print(tbl.head(30).to_string(index=False))



In [ ]:
# === NEW CELL: convert J$ -> US$ and print a tidy table ======================
FX_JMD_PER_USD = 150.0  # 1 USD = 150 JMD (your fixed rate)

# Columns to convert
cols = ["avoided_ead_min", "avoided_ead_max", "avoided_ead_mid"]

result = result.copy()
for c in cols:
    result[f"{c}_usd"]      = result[c] / FX_JMD_PER_USD
    result[f"{c}_usd_mn"]   = result[c] / FX_JMD_PER_USD / 1e6
    result[f"{c}_usd_bil"]  = result[c] / FX_JMD_PER_USD / 1e9

# Neat table in USD millions, sorted by max
tbl_usd = (
    result[["catchment_uid"] + [f"{c}_usd_mn" for c in cols]]
      .rename(columns={
          "avoided_ead_min_usd_mn": "min_usd_mn",
          "avoided_ead_max_usd_mn": "max_usd_mn",
          "avoided_ead_mid_usd_mn": "mid_usd_mn",
      })
      .sort_values("max_usd_mn", ascending=False)
)

# Print top 30 with sensible rounding
pd.options.display.float_format = "{:,.1f}".format
print("Columns:", list(tbl_usd.columns))
print(tbl_usd.head(30).to_string(index=False))

# (Optional) CSV export
out_csv = output_dir / "catchment_avoided_ead_usd_millions_top.csv"
tbl_usd.to_csv(out_csv, index=False); print("Saved:", out_csv)

In [ ]:
# # values in billions
# result["_ead_bil"] = (result["avoided_ead_max"] / 1e9).astype(float)
# vals = result["_ead_bil"].fillna(0).to_numpy()
# hi = np.percentile(vals[vals > 0], 98) if (vals > 0).any() else float(vals.max())

# TITLE_FS = 7; LABEL_FS = 6; TICK_FS = 5.5; LINE_W = 0.35
# TITLE = "Maximum avoided expected annual damages (EADs) by major catchment"
# TITLE_WRAPPED = "Maximum avoided expected annual damages (EADs)\nby major catchment"  # wrap to 2 lines

# with mpl.rc_context({"font.family": "Arial", "axes.linewidth": LINE_W, "figure.dpi": 300}):
#     # make it a touch taller to give the title breathing room
#     fig, ax = plt.subplots(figsize=(mm_to_in(85), mm_to_in(62)), constrained_layout=False)
#     ax.set_axis_off()

#     m = result.plot(
#         ax=ax, column="_ead_bil",
#         vmin=0, vmax=(hi if hi and hi > 0 else None),
#         cmap="viridis", edgecolor="white", linewidth=0.3
#     )

#     # colorbar (slim, small text)
#     norm = mpl.colors.Normalize(vmin=0, vmax=(hi if hi and hi > 0 else None))
#     sm = mpl.cm.ScalarMappable(norm=norm, cmap="viridis"); sm._A = []
#     cbar = fig.colorbar(sm, ax=ax, fraction=0.028, pad=0.012)
#     cbar.outline.set_linewidth(LINE_W)
#     cbar.ax.tick_params(width=LINE_W, length=2.0, labelsize=TICK_FS)
#     cbar.set_label("Avoided EAD (J$) in billions", fontsize=LABEL_FS, labelpad=4)

#     # leave space for title (top) and colorbar (right)
#     fig.subplots_adjust(top=0.86, right=0.86)

#     # figure-level title (not ax.set_title) so it won't squeeze the map
#     fig.suptitle(TITLE_WRAPPED, fontsize=TITLE_FS, y=0.96)

#     out_png = output_dir / "figure_avoided_ead_max_nature_billions.png"
#     out_pdf = output_dir / "figure_avoided_ead_max_nature_billions.pdf"
#     fig.savefig(out_png, dpi=600, bbox_inches="tight")
#     fig.savefig(out_pdf, bbox_inches="tight")
#     plt.show()

# print("Saved:", out_png, "and", out_pdf)

In [ ]:
# Intersections of catchments with Admin-1 (PARISH)
intersections = gpd.overlay(
    result[["catchment_uid", "geometry"]],
    admin1[["PARISH", "geometry"]],
    how="intersection"
).copy()

# Areas
intersections["part_m2"] = intersections.geometry.area
catch_area_m2 = result.set_index("catchment_uid").geometry.area

# Summarize shares per (catchment, PARISH)
grp = (intersections
       .groupby(["catchment_uid", "PARISH"], as_index=False)
       .agg(part_m2=("part_m2", "sum")))

grp["catch_area_m2"] = grp["catchment_uid"].map(catch_area_m2)
shares = grp.assign(
    area_km2 = grp["part_m2"] / 1e6,
    share    = grp["part_m2"] / grp["catch_area_m2"],
    share_pct = lambda d: (d["share"] * 100).round(1).clip(0, 100)
).drop(columns="part_m2")

intersections

In [ ]:
# Count distinct parishes per catchment
n_parishes = (shares.groupby("catchment_uid")["PARISH"]
                      .nunique()
                      .rename("n_parishes")
                      .reset_index())

# Distribution of counts
print("Parish-count distribution:\n", n_parishes["n_parishes"].value_counts().sort_index())

# Top examples with >1 parish
print("\nCatchments crossing >1 parish (top 10):")
print(n_parishes[n_parishes["n_parishes"] > 1]
      .sort_values("n_parishes", ascending=False)
      .head(10).to_string(index=False))

In [ ]:
# Parish % list for the top 10 catchments in your avoided_ead_max table
top_ids = tbl["catchment_uid"].head(10).tolist()

brk = (shares[shares["catchment_uid"].isin(top_ids)]
       .sort_values(["catchment_uid", "share"], ascending=[True, False])
       .assign(entry=lambda d: d["PARISH"] + " (" + d["share_pct"].astype(str) + "%)")
       .groupby("catchment_uid")["entry"]
       .agg(", ".join)
       .reset_index(name="parish_breakdown"))

print(brk.to_string(index=False))

brk

# Save the top-10 parish breakdown to CSV
out_csv = output_dir / "top10_parish_breakdown_by_catchment.csv"
brk.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [ ]:
# === Parish breakdowns for *every* catchment =================================

# 1) Compact per-catchment breakdown string (e.g., "ParishA (62.3%), ParishB (37.7%)")
shares_for_txt = shares.copy()  # use shares_nosliver if you filtered tiny slivers
shares_for_txt["share_pct"] = shares_for_txt["share_pct"].round(1)

brk_all = (shares_for_txt
           .sort_values(["catchment_uid", "share"], ascending=[True, False])
           .assign(entry=lambda d: d["PARISH"] + " (" + d["share_pct"].astype(str) + "%)")
           .groupby("catchment_uid", as_index=False)["entry"]
           .agg(", ".join)
           .rename(columns={"entry": "parish_breakdown"}))

# 2) Majority parish (and add area if you have it)
maj = (shares_for_txt.loc[shares_for_txt.groupby("catchment_uid")["share"].idxmax(),
                          ["catchment_uid", "PARISH", "share_pct"]]
       .rename(columns={"PARISH": "majority_parish",
                        "share_pct": "majority_share_pct"}))

areas = (result[["catchment_uid","area_km2"]]
         if "area_km2" in result.columns
         else result[["catchment_uid"]].assign(area_km2=result.geometry.area/1e6))

out = (brk_all
       .merge(maj, on="catchment_uid", how="left")
       .merge(areas, on="catchment_uid", how="left")
       .sort_values("catchment_uid"))

# 3) Save CSVs
out_csv = output_dir / "catchment_parish_breakdown_all.csv"
out.to_csv(out_csv, index=False)
print("Saved:", out_csv)

shares_long_csv = output_dir / "catchment_parish_shares_long_all.csv"
(shares[["catchment_uid","PARISH","area_km2","share_pct"]]
 .sort_values(["catchment_uid","share_pct"], ascending=[True, False])
 .to_csv(shares_long_csv, index=False))
print("Saved:", shares_long_csv)

wide_csv = output_dir / "catchment_parish_shares_wide_all.csv"
(shares.pivot_table(index="catchment_uid", columns="PARISH", values="share_pct", fill_value=0)
 .round(1).sort_index(axis=1).to_csv(wide_csv))
print("Saved:", wide_csv)

In [ ]:
# n_zero = (result["avoided_ead_max"] == 0).sum()
# n_nan  = result["avoided_ead_max"].isna().sum()
# min_pos = result.loc[result["avoided_ead_max"] > 0, "avoided_ead_max"].min()

# print(f"zeros: {n_zero}, NaNs: {n_nan}, smallest positive: {min_pos:.3g}")
# print("IDs with zero:", result.loc[result["avoided_ead_max"] == 0, "catchment_uid" if "catchment_uid" in result else "basin_id"].tolist())

In [ ]:
def stats(df, col):
    n = len(df)
    v = df[col]
    pos = (v > 0).sum()
    zero = (v == 0).sum()
    neg = (v < 0).sum()   # just in case
    nan = v.isna().sum()
    return pd.Series({
        "catchments_total": n,
        "positive_count": int(pos),
        "zero_count": int(zero),
        "negative_count": int(neg),
        "nan_count": int(nan),
        "positive_share_pct": round(100 * pos / n, 1),
        "zero_share_pct": round(100 * zero / n, 1),
    })

summary = pd.DataFrame({
    "avoided_ead_min": stats(result, "avoided_ead_min"),
    "avoided_ead_max": stats(result, "avoided_ead_max"),
}).T.reset_index().rename(columns={"index": "metric"})

print(summary.to_string(index=False))

In [ ]:
print("catchments has catchment_uid:", "catchment_uid" in catchments.columns)
print("result has catchment_uid:", "catchment_uid" in result.columns)
print("Null UIDs in catchments:", catchments["catchment_uid"].isna().sum())
print("Unique UIDs in catchments:", catchments["catchment_uid"].nunique())
print(catchments[["catchment_uid"]].head())

# I DON'T THINK I SHOULD USE SMOOTHED

In [ ]:
# === POWER-STRETCH MAP (continuous, with tidy colorbar ticks) ================


reproj = avoided_fluvial_ead_max_300m_smoothed  # your 300 m raster

def smart_unit(jmax):
    """Pick display units so the upper cap lands in a readable 1–1000 range."""
    for sc, lab in [(1e-9,"J$ billions"), (1e-6,"J$ millions"), (1e-3,"J$ thousands"), (1.0,"J$")]:
        y = jmax * sc
        if 1 <= y < 1000:
            return sc, lab
    return (1.0, "J$")

# Read raster and extent
with rasterio.open(reproj) as src:
    A = src.read(1, masked=True)   # masked array (respects NoData)
    extent = plotting_extent(src)

# Positive-only stats
pos = A[(~A.mask) & (A > 0)]
if pos.size == 0:
    raise ValueError("No positive values to plot.")

# Cap top to reveal mid-tones and choose display units
hi = float(np.percentile(pos, 99.5))
scale, unit_label = smart_unit(hi)

# Layers: zeros underlay; positives on top
A_pos = A.copy()
A_pos[A_pos <= 1] = np.nan  # keep your thresholding exactly as written

# Power/gamma stretch (you chose linear here)
# norm = PowerNorm(gamma=0.45, vmin=0.1, vmax=hi)
norm = mpl.colors.Normalize(vmin=0, vmax=hi)
cmap = mpl.colormaps["magma_r"]  # near-white → color (print-friendly)

with mpl.rc_context(NATURE_RC):
    fig, ax = plt.subplots(figsize=(mm_to_in(90), mm_to_in(60)))
    im = ax.imshow(A_pos, norm=norm, cmap=cmap, extent=extent, origin="upper", interpolation="nearest")
    ax.set_axis_off()

    # Boundary (optional)
    outline_gdf = None
    try:
        outline = (jamaica_boundary.union_all() if hasattr(jamaica_boundary, "union_all")
                   else jamaica_boundary.unary_union)
        outline_gdf = gpd.GeoSeries([outline], crs=jamaica_boundary.crs)
        outline_gdf.plot(ax=ax, facecolor="none", edgecolor="white", lw=1.0, zorder=5)
        outline_gdf.plot(ax=ax, facecolor="none", edgecolor="black", lw=0.45, zorder=6)
    except NameError:
        pass

    # --- Scale bar & north arrow (tries your helpers; falls back if missing) ---
    try:
        # Use your high-quality helpers if available
        if outline_gdf is not None:
            pt = draw_scale_bar(
                ax, outline_gdf, where="right-top",
                pad=0.07, length_km="auto", max_frac=0.22,
                lw=0.5, tick_h_frac=0.012, fs_lab=5, fs_unit=5, unit_text="km"
            )
            if pt is not None:
                cx_data, cy_data = pt
                cx_ax, cy_ax = ax.transAxes.inverted().transform(ax.transData.transform((cx_data, cy_data)))
                draw_north_arrow_axes(
                    ax, cx_ax, cy_ax,
                    size_frac=0.080, gap_frac=0.050,
                    shaft_w_frac=0.10, head_w_frac=0.30, head_h_frac=0.50,
                    fs=5, lw=0.5
                )
    except NameError:
        # # Fallback: simple in-axes versions
        # def add_scale_bar(ax, length_km=25, location=(0.62, 0.83), lw=0.6, fs=6):
        #     half = 0.05; x, y = location
        #     ax.plot([x-half, x+half], [y, y], transform=ax.transAxes, color='black', lw=lw)
        #     for pos_ in (x-half, x, x+half):
        #         ax.plot([pos_, pos_], [y-0.006, y+0.006], transform=ax.transAxes, color='black', lw=lw)
        #     ax.text(x-half, y-0.03, "0", transform=ax.transAxes, ha='center', va='center', fontsize=fs)
        #     ax.text(x,       y-0.03, f"{int(length_km//2)}", transform=ax.transAxes, ha='center', va='center', fontsize=fs)
        #     ax.text(x+half,  y-0.03, f"{int(length_km)}",    transform=ax.transAxes, ha='center', va='center', fontsize=fs)
        #     ax.text(x+half+0.02, y, "km", transform=ax.transAxes, ha='left', va='center', fontsize=fs)

        # def add_north_arrow(ax, location=(0.76, 0.88), size=0.05, fs=6):
        #     x, y = location
        #     ax.annotate(
        #         "", xy=(x, y+size), xycoords='axes fraction',
        #         xytext=(x, y), textcoords='axes fraction',
        #         arrowprops=dict(facecolor='black', edgecolor='black', headwidth=6, headlength=8, width=2)
        #     )
        #     ax.text(x, y+size+0.015, "N", transform=ax.transAxes,
        #             ha='center', va='center', fontsize=fs, fontweight='bold')

        add_scale_bar(ax)
        add_north_arrow(ax)

    # Colorbar with clean, rounded ticks in the chosen unit
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.024, pad=0.012)
    cbar.ax.yaxis.set_minor_locator(NullLocator())
    cbar.outline.set_linewidth(0.35)

    # Compute nice ticks (4–6 ticks, 1–2–2.5–5–10 steps) in display units
    hi_scaled = hi * scale
    locator = MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10], min_n_ticks=4)
    ticks_scaled = locator.tick_values(0, hi_scaled)
    ticks_scaled = ticks_scaled[(ticks_scaled >= 0) & (ticks_scaled <= hi_scaled + 1e-12)]
    ticks_raw = ticks_scaled / scale  # convert back to raw units for placement
    cbar.set_ticks(ticks_raw)

    def _fmt_label(val_raw):
        y = val_raw * scale
        if y < 1:   s = f"{y:.2f}"
        elif y < 10: s = f"{y:.1f}"
        else:        s = f"{y:.0f}"
        return s.rstrip("0").rstrip(".")
    cbar.set_ticklabels([_fmt_label(t) for t in ticks_raw])
    cbar.set_label(f"Avoided EADs ({unit_label})", fontsize=6)

    fig.subplots_adjust(top=0.965, right=0.86)
    fig.suptitle("Fig. 3(a): Locations of avoided EADs from forest restoration",
                 fontsize=7, y=0.985)

    out_png = output_dir / "avoided_fluvial_ead_300m_powernorm.png"
    # out_pdf = output_dir / "avoided_fluvial_ead_300m_powernorm.pdf"
    fig.savefig(out_png, dpi=600, bbox_inches="tight")
    fig.savefig(out_pdf,              bbox_inches="tight")
    plt.show()
    print("Saved:", out_png, "and", out_pdf)